# PSM Experiment — Kaggle Notebook

**Predictive Semantic Memory (PSM)** experiment runner for Kaggle.

This notebook:
1. Loads a Kaggle-hosted LLM (no external API key needed)
2. Runs a **smoke test** (synthetic questions) to validate the pipeline end-to-end
3. Runs the **full TriviaQA experiment** (smoke + pilot profiles)
4. Saves **all results as human-readable CSV files** ready to push to GitHub

---
### ⚡ Quick-start
1. Open **Notebook settings → Models** and add your preferred model  
   *(Qwen2 7B Instruct GPTQ-INT4 is a great choice — accept the Kaggle licence once)*
2. Set `KAGGLE_MODEL_PATH` in the **Config** cell below to match the path Kaggle assigns
3. Run all cells top-to-bottom (**Run → Run All**)


## 1 · Install required packages

Kaggle pre-installs `torch` and `transformers`. The cells below install the few extra packages the PSM pipeline needs.

In [ ]:
%%capture install_output
import subprocess, sys

PACKAGES = [
    "transformers==5.4.0",
    "huggingface_hub",
    "safetensors",
    "accelerate",
    "sentencepiece",
    "faiss-cpu",
    "sentence-transformers",
    "rank-bm25",
    "datasets",
    "rouge-score",
    "nltk",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])
print("✓ All packages installed.")


In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("✓ NLTK data ready.")


## 2 · Configuration

Edit the variables below to match **your** Kaggle notebook setup.

### Common Kaggle model paths (after adding the model in notebook settings)
| Model | Kaggle path |
|-------|-------------|
| **Gemma 3 4B IT** | `/kaggle/input/gemma-3/transformers/4b-it/1` |
| **Gemma 2 2B IT** *(safe default)* | `/kaggle/input/gemma-2/transformers/2b-it/1` |
| **Qwen2 7B Instruct GPTQ-INT4** *(recommended)* | `/kaggle/input/qwen-lm/qwen2/transformers/7b-instruct-gptq-int4/1` |
| Gemma 7B IT | `/kaggle/input/gemma/transformers/7b-it/3` |
| Mistral 7B Instruct v0.2 | `/kaggle/input/mistral-7b-instruct-v0.2/transformers/default/1` |
| Llama 3 8B Instruct | `/kaggle/input/llama-3/transformers/8b-instruct/1` |

> **Tip:** After adding a model, click the ⓘ icon next to its name to see the exact path.


In [ ]:
import os
import json
from pathlib import Path

# Optional manual override: set this to an exact attached model path if needed.
# Example: "/kaggle/input/gemma-2/transformers/2b-it/1"
KAGGLE_MODEL_PATH = os.environ.get("PSM_KAGGLE_MODEL_PATH", "").strip()

# Try common attached Kaggle model mount paths (first existing path wins).
# Gemma 2 2B IT is the safest default for Kaggle runs.
MODEL_CANDIDATES = [
    "/kaggle/input/gemma-2/transformers/2b-it/1",
    "/kaggle/input/gemma-3/transformers/4b-it/1",
    "/kaggle/input/gemma/transformers/2b-it/3",
    "/kaggle/input/gemma/transformers/7b-it/3",
    "/kaggle/input/mistral-7b-instruct-v0.2/transformers/default/1",
    "/kaggle/input/llama-3/transformers/8b-instruct/1",
    "/kaggle/input/qwen-lm/qwen2/transformers/7b-instruct-gptq-int4/1",
]

SUPPORTED_MODEL_TYPES = {"gemma2", "gemma3", "gemma", "phi", "llama", "mistral", "qwen2"}

def read_model_type(model_path: str):
    cfg = Path(model_path) / "config.json"
    if not cfg.exists():
        return None
    try:
        with cfg.open("r", encoding="utf-8") as f:
            data = json.load(f)
        return data.get("model_type")
    except Exception:
        return None

def is_supported_model_path(model_path: str) -> bool:
    if not Path(model_path).exists():
        return False
    model_type = read_model_type(model_path)
    return model_type in SUPPORTED_MODEL_TYPES

# Public fallback model id from Hugging Face Hub (used when no Kaggle model is attached).
HF_FALLBACK_MODEL_ID = "google/gemma-2-2b-it"

if KAGGLE_MODEL_PATH and is_supported_model_path(KAGGLE_MODEL_PATH):
    resolved_model_path = KAGGLE_MODEL_PATH
else:
    resolved_model_path = next((p for p in MODEL_CANDIDATES if is_supported_model_path(p)), None)

# If no local Kaggle model mount found, fall back to the public HF model id and prepare to download.
using_hf_fallback = False
if resolved_model_path is None:
    resolved_model_path = HF_FALLBACK_MODEL_ID
    using_hf_fallback = True

# If the resolved path is a local mount, try to detect model type from config, otherwise mark as 'hf_fallback'.
if using_hf_fallback:
    SELECTED_MODEL_TYPE = "hf_fallback"
else:
    SELECTED_MODEL_TYPE = read_model_type(resolved_model_path)

KAGGLE_MODEL_PATH = resolved_model_path

# -- Experiment settings -------------------------------------------------------
SMOKE_SIZE = 10
CONFIDENCE_THRESHOLD = 0.55

# -- Output directory ----------------------------------------------------------
OUTPUT_BASE = "/kaggle/working/psm_results"

# Apply environment variables so the PSM pipeline picks them up
os.environ["PSM_LLM_BACKEND"] = "kaggle"
os.environ["PSM_KAGGLE_MODEL_PATH"] = KAGGLE_MODEL_PATH
os.environ["PSM_NUM_PREDICT"] = "64"
os.environ["PSM_TEMPERATURE"] = "0.1"

print(f"LLM backend  : {os.environ['PSM_LLM_BACKEND']}")
print(f"Model path   : {os.environ['PSM_KAGGLE_MODEL_PATH']}")
print(f"Model type   : {SELECTED_MODEL_TYPE}")
print(f"Using HF fallback: {using_hf_fallback}")
print(f"Output base  : {OUTPUT_BASE}")
print("✓ Model path verified and configuration applied.")

In [ ]:
# If we are using a HF Hub id (fallback) download the model locally so transformers can load it.
# This runs only when no Kaggle model mount is attached and internet access is available.
try:
    from pathlib import Path
    from huggingface_hub import snapshot_download
    
    if os.environ.get("PSM_KAGGLE_MODEL_PATH", "").count("/") >= 1 and SELECTED_MODEL_TYPE == "hf_fallback":
        hf_id = os.environ["PSM_KAGGLE_MODEL_PATH"]
        download_dir = Path("/kaggle/working/hf_models") / hf_id.replace("/", "_")
        download_dir.mkdir(parents=True, exist_ok=True)
        print(f"Downloading Hugging Face model {hf_id} into {download_dir} (this may take several minutes)...")
        snapshot_download(repo_id=hf_id, repo_type="model", local_dir=str(download_dir), local_dir_use_symlinks=False)
        # Point the pipeline to the downloaded snapshot
        os.environ["PSM_KAGGLE_MODEL_PATH"] = str(download_dir)
        KAGGLE_MODEL_PATH = str(download_dir)
        SELECTED_MODEL_TYPE = read_model_type(KAGGLE_MODEL_PATH) or "hf_download"
        print("✓ HF model downloaded and environment updated.")
    else:
        print("No HF fallback download needed.")
except Exception as exc:
    print(f"Warning: HF fallback download failed: {exc}")
    print("If you have attached a Kaggle Model, re-run the config cell to pick it up.")


## 3 · Import PSM modules

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def is_psm_root(path: Path) -> bool:
    return (
        (path / "router" / "router.py").exists()
        and (path / "llm").exists()
        and (path / "run_psm_experiments.py").exists()
    )


# 1) Fast explicit candidates (dataset mounts + cwd)
PSM_ROOT = None
candidates = [
    Path("/kaggle/input/datasets/kausikvaibhavpatra/psm-experiment/PSM_Experiment_Setup_Research"),
    Path("/kaggle/input/psm-experiment-research-setup-updated/PSM_Experiment_Setup_Research"),
    Path("/kaggle/input/psm-experiment/PSM_Experiment_Setup_Research"),
    Path.cwd() / "PSM_Experiment_Setup_Research",
    Path.cwd(),
]

for candidate in candidates:
    if is_psm_root(candidate):
        PSM_ROOT = candidate
        break

# 2) Recursive scan under /kaggle/input (handles nested dataset layouts)
if PSM_ROOT is None:
    for router_file in Path("/kaggle/input").rglob("router.py"):
        possible_root = router_file.parent.parent
        if is_psm_root(possible_root):
            PSM_ROOT = possible_root
            break

# 3) Fallback: clone repo into /kaggle/working when no dataset is attached
if PSM_ROOT is None:
    repo_url = "https://github.com/KausikVP30/PSM_Experiment_Research_Setup_Updated.git"
    clone_dir = Path("/kaggle/working/PSM_Experiment_Research_Setup_Updated")

    if not clone_dir.exists():
        print("[INFO] No attached PSM dataset found. Cloning repo fallback...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", repo_url, str(clone_dir)], check=True)
    else:
        print("[INFO] Refreshing fallback repo to latest origin/main...")
        subprocess.run(["git", "-C", str(clone_dir), "fetch", "--depth", "1", "origin", "main"], check=False)
        subprocess.run(["git", "-C", str(clone_dir), "checkout", "main"], check=False)
        subprocess.run(["git", "-C", str(clone_dir), "reset", "--hard", "origin/main"], check=False)

    fallback_candidates = [
        clone_dir / "PSM_Experiment_Setup_Research",
        clone_dir / "PSM_Experiment_Research_Setup_Updated-main" / "PSM_Experiment_Research_Setup_Updated-main" / "PSM_Experiment_Setup_Research",
        clone_dir,
    ]

    for candidate in fallback_candidates:
        if is_psm_root(candidate):
            PSM_ROOT = candidate
            break

if PSM_ROOT is None:
    raise RuntimeError(
        "PSM source tree not found after checking mounted datasets and clone fallback.\n"
        "Attach a dataset containing PSM_Experiment_Setup_Research or keep internet enabled for clone fallback."
    )

sys.path.insert(0, str(PSM_ROOT))
print(f"PSM root: {PSM_ROOT}")

# Verify core imports
from router.router import Router
from evaluation.metrics import exact_match, token_f1, rouge_l, bleu, contains_any_answer
from run_psm_experiments import PSMExperimentRunner, ExperimentProfile
from triviaqa_smoke_run_fast import PSMSmokeRunFast

print("✓ All PSM modules imported successfully.")

## 4 · Smoke test (synthetic questions)

This cell runs **10 synthetic questions** through the full pipeline:
*embedding → FAISS retrieval → LLM generation → metrics*.

It must pass before the full TriviaQA experiment runs.  
✅ Pass = all 10 questions complete without errors.


In [ ]:
import os, shutil
from pathlib import Path
import pandas as pd

SMOKE_OUT = Path(OUTPUT_BASE) / "smoke_test"
shutil.rmtree(SMOKE_OUT, ignore_errors=True)   # fresh run each time

# PSMSmokeRunFast writes its outputs to smoke_results/<run_id>/<timestamp>/
# We change cwd temporarily so relative paths land inside our output folder
original_cwd = os.getcwd()
os.makedirs(SMOKE_OUT, exist_ok=True)
os.chdir(SMOKE_OUT)

try:
    smoke = PSMSmokeRunFast(smoke_size=SMOKE_SIZE, run_id="kaggle_smoke")
    smoke_metrics = smoke.run()
finally:
    os.chdir(original_cwd)

if smoke_metrics is None:
    raise RuntimeError("Smoke test produced no metrics — check errors above.")

print()
print("=" * 60)
print("SMOKE TEST RESULT")
print("=" * 60)
print(f"  Total questions : {smoke_metrics['total_queries']}")
print(f"  Avg EM          : {smoke_metrics['avg_em']:.3f}")
print(f"  Avg F1          : {smoke_metrics['avg_f1']:.3f}")
print(f"  Avg Confidence  : {smoke_metrics['avg_confidence']:.3f}")
print(f"  Memory rate     : {smoke_metrics['memory_path_rate']:.1%}")
print(f"  Retrieval rate  : {smoke_metrics['retrieval_path_rate']:.1%}")
print("=" * 60)
print("✓ SMOKE TEST PASSED — pipeline is healthy, proceeding to full experiment.")


### 4a · Smoke test results (CSV preview)

In [ ]:
import glob, pandas as pd

smoke_pred_files = glob.glob(str(SMOKE_OUT / "**" / "predictions.jsonl"), recursive=True)

if smoke_pred_files:
    smoke_df = pd.read_json(smoke_pred_files[0], lines=True)
    print(f"Predictions shape: {smoke_df.shape}")
    display(smoke_df[["query_id", "question", "generated_answer", "correct_answer",
                       "em", "f1", "confidence", "source"]].head(SMOKE_SIZE))
else:
    print("No predictions file found (check smoke test cell for errors).")


## 5 · Full experiment — TriviaQA

Runs two profiles back-to-back:

| Profile | Questions | Corpus pool | Description |
|---------|-----------|-------------|-------------|
| `smoke` | 5 | 140 docs | Fast sanity check on real data |
| `pilot` | 8 | 260 docs | Small but realistic evaluation |

All results are written to `OUTPUT_BASE/experiment_runs/` as:
- `predictions.csv` — per-question results with all metrics
- `summary_metrics.csv` — flat one-row summary (easy to open in Excel)
- `summary_metrics.json` — full nested summary
- `all_runs_metrics.csv` — aggregated table across all profiles


In [ ]:
import os, shutil
from pathlib import Path

EXP_OUT = Path(OUTPUT_BASE) / "experiment_runs"
shutil.rmtree(EXP_OUT, ignore_errors=True)
EXP_OUT.mkdir(parents=True, exist_ok=True)

original_cwd = os.getcwd()
os.chdir(Path(OUTPUT_BASE))   # runner uses relative paths internally

try:
    runner = PSMExperimentRunner(
        base_results_dir=str(EXP_OUT),
        threshold=CONFIDENCE_THRESHOLD,
    )
    experiment_results = runner.execute()
finally:
    os.chdir(original_cwd)

print()
print("=" * 60)
print("EXPERIMENT COMPLETE")
print(f"  Runs completed : {len(experiment_results['runs'])}")
print(f"  Output dir     : {EXP_OUT}")
print("=" * 60)


## 6 · Evaluation metrics — all CSV results

In [ ]:
import glob, pandas as pd
from pathlib import Path

EXP_OUT = Path(OUTPUT_BASE) / "experiment_runs"

# ── Per-run predictions ──────────────────────────────────────────────────────
pred_csvs = sorted(EXP_OUT.glob("**/predictions.csv"))
print(f"Found {len(pred_csvs)} predictions.csv file(s)")
print()

all_preds = []
for p in pred_csvs:
    run_name = p.parent.name
    df = pd.read_csv(p)
    df["run_name"] = run_name
    all_preds.append(df)
    print(f"─── {run_name}  ({len(df)} rows) ───")
    print(df[["query_id", "question", "generated_answer", "correct_answer",
              "em", "f1", "rouge_l", "bleu", "confidence", "source",
              "latency_s"]].to_string(index=False, max_colwidth=55))
    print()

if all_preds:
    combined_preds = pd.concat(all_preds, ignore_index=True)
    combined_path = EXP_OUT / "all_predictions_combined.csv"
    combined_preds.to_csv(combined_path, index=False)
    print(f"✓ Combined predictions saved → {combined_path}")


In [ ]:
# ── Summary metrics (flat CSV, one row per run) ──────────────────────────────
summary_csvs = sorted(EXP_OUT.glob("**/summary_metrics.csv"))
print(f"Found {len(summary_csvs)} summary_metrics.csv file(s)")
print()

all_summaries = []
for s in summary_csvs:
    df = pd.read_csv(s)
    all_summaries.append(df)

if all_summaries:
    summary_df = pd.concat(all_summaries, ignore_index=True)
    cols_to_show = [
        "run_id", "num_questions", "avg_em", "avg_f1", "avg_rouge_l", "avg_bleu",
        "avg_confidence", "avg_latency_s", "hallucination_rate",
        "memory_path_rate", "retrieval_path_rate", "final_memory_size",
    ]
    print(summary_df[cols_to_show].to_string(index=False))
    print()

# ── Master aggregate CSV ─────────────────────────────────────────────────────
master_csv = EXP_OUT / "all_runs_metrics.csv"
if master_csv.exists():
    master_df = pd.read_csv(master_csv)
    print("─── all_runs_metrics.csv (aggregate) ───")
    print(master_df.to_string(index=False))


## 7 · Output files — ready to push to GitHub

In [ ]:
import os
from pathlib import Path

OUTPUT_BASE_PATH = Path(OUTPUT_BASE)

print("=" * 70)
print("ALL OUTPUT FILES")
print("=" * 70)

csv_files = sorted(OUTPUT_BASE_PATH.rglob("*.csv"))
json_files = sorted(OUTPUT_BASE_PATH.rglob("*.json"))
txt_files  = sorted(OUTPUT_BASE_PATH.rglob("*.txt"))

print(f"\n📊  CSV files ({len(csv_files)}) — human-readable, GitHub-friendly:")
for f in csv_files:
    size_kb = f.stat().st_size / 1024
    print(f"    {f.relative_to(OUTPUT_BASE_PATH)}  ({size_kb:.1f} KB)")

print(f"\n📋  JSON files ({len(json_files)}):")
for f in json_files:
    size_kb = f.stat().st_size / 1024
    print(f"    {f.relative_to(OUTPUT_BASE_PATH)}  ({size_kb:.1f} KB)")

print(f"\n📝  Text reports ({len(txt_files)}):")
for f in txt_files:
    size_kb = f.stat().st_size / 1024
    print(f"    {f.relative_to(OUTPUT_BASE_PATH)}  ({size_kb:.1f} KB)")

print()
print("=" * 70)
print("To push to GitHub:")
print("  1. Download the output folder from Kaggle (⬇ Output tab)")
print("  2. Copy the CSVs into your repo under results/<run_id>/")
print("  3. git add results/ && git commit -m 'Add experiment results' && git push")
print("=" * 70)


## 8 · (Optional) Larger experiment

Uncomment and adjust the cell below to run a larger evaluation.  
Increase `num_questions` and `corpus_pool_size` to match your Kaggle session GPU quota.


In [ ]:
# # Uncomment to run a larger experiment
#
# from run_psm_experiments import PSMExperimentRunner, ExperimentProfile
# import os
# from pathlib import Path
#
# LARGE_OUT = Path(OUTPUT_BASE) / "large_experiment"
# LARGE_OUT.mkdir(parents=True, exist_ok=True)
# original_cwd = os.getcwd()
# os.chdir(Path(OUTPUT_BASE))
#
# try:
#     runner = PSMExperimentRunner(
#         base_results_dir=str(LARGE_OUT),
#         threshold=CONFIDENCE_THRESHOLD,
#     )
#     # Override profiles for a larger run
#     profiles = [
#         ExperimentProfile(name="medium",  num_questions=25,  corpus_pool_size=500,  max_chunks=1500),
#         ExperimentProfile(name="full",    num_questions=100, corpus_pool_size=1000, max_chunks=3000),
#     ]
#     for p in profiles:
#         runner.run_profile(p)
# finally:
#     os.chdir(original_cwd)
